# Apicurio Registry API Compatibility Probe

This notebook checks Apicurio's Confluent-compatible v7 API first, then uses the Core v3 API for the richer Apicurio-native metadata needed by the StreamBridge UI.

## 1. Set Up Notebook and Imports

The notebook uses `requests` for HTTP calls, environment variables for deployment-specific settings, and display helpers for readable output.

In [ ]:
import json
import logging
import os
from typing import Any
from urllib.parse import quote

import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
LOGGER = logging.getLogger("apicurio-probe")
NOTEBOOK_DIR = "sandbox/ec2/schema_registry"
REQUEST_TIMEOUT_SECONDS = float(os.getenv("APICURIO_TIMEOUT_SECONDS", "10"))


## 2. Configure Registry API Base URL and Headers

Set `APICURIO_REGISTRY_URL` to the registry origin, for example `http://localhost:8080`. The notebook tests the Confluent-compatible v7 API first, while Core v3 remains the preferred native API for detailed metadata.

In [10]:
REGISTRY_URL = os.getenv("APICURIO_REGISTRY_URL", "http://localhost:8081").rstrip("/")
API_PREFIX = "/apis/registry/v3"
API_VARIANTS = {
    "core_v3": "/apis/registry/v3",
    "core_v2_deprecated": "/apis/registry/v2",
    "confluent_v7_compatibility": "/apis/ccompat/v7",
}

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
}
access_token = os.getenv("APICURIO_ACCESS_TOKEN")
if access_token:
    headers["Authorization"] = f"Bearer {access_token}"


def api_url(path: str, api_prefix: str = API_PREFIX) -> str:
    """Build a URL using an explicit Apicurio API prefix."""
    operation_path = "/" + path.lstrip("/")
    prefix = "/" + api_prefix.strip("/")
    if operation_path.startswith(prefix + "/") or operation_path == prefix:
        return f"{REGISTRY_URL}{operation_path}"
    return f"{REGISTRY_URL}{prefix}{operation_path}"


print(f"Registry: {REGISTRY_URL}")
print(f"Preferred API: {API_PREFIX}")
print(f"Compatibility APIs: {API_VARIANTS['core_v2_deprecated']}, {API_VARIANTS['confluent_v7_compatibility']}")


Registry: http://localhost:8081
Preferred API: /apis/registry/v3
Compatibility APIs: /apis/registry/v2, /apis/ccompat/v7


## API Variants

The v7 compatibility API is tested first because it can provide one common contract for Apicurio and Confluent. Core v3 is then used for Apicurio-specific artifact metadata. Core v2 is deprecated and is not used by this probe.

In [12]:
def request_json(
    path: str,
    *,
    params: dict[str, Any] | None = None,
    retries: int = 2,
    api_prefix: str = API_PREFIX,
) -> Any:
    """GET one Apicurio operation with timeout, retry, and useful failure details."""
    url = api_url(path, api_prefix)
    last_error: Exception | None = None
    for attempt in range(retries + 1):
        try:
            response = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.ok:
                if not response.content:
                    return None
                content_type = response.headers.get("content-type", "")
                return response.json() if "json" in content_type else response.text
            detail = response.text[:500]
            raise requests.HTTPError(f"HTTP {response.status_code} for {url}: {detail}")
        except (requests.RequestException, ValueError) as error:
            last_error = error
            LOGGER.warning("Attempt %s failed for %s: %s", attempt + 1, url, error)
            if attempt == retries:
                break
    raise RuntimeError(f"Registry request failed: {url}") from last_error


def show_result(label: str, value: Any) -> None:
    print(label)
    print(json.dumps(value, indent=2, default=str))


## 3. Test Confluent-Compatible v7 API First

These are the standard Confluent Schema Registry operations exposed by Apicurio: subjects, subject versions, latest schema, and compatibility configuration.

In [13]:
V7_PREFIX = API_VARIANTS["confluent_v7_compatibility"]


def probe_confluent_v7() -> dict[str, Any]:
    result: dict[str, Any] = {
        "apiVersion": "v7",
        "apiPrefix": V7_PREFIX,
        "compatible": False,
        "subjects": [],
        "config": None,
        "errors": [],
    }

    try:
        subjects = request_json("/subjects", api_prefix=V7_PREFIX) or []
        result["subjects"] = subjects if isinstance(subjects, list) else []
        result["compatible"] = True
    except RuntimeError as error:
        result["errors"].append({"operation": "subjects", "message": str(error)})
        return result

    try:
        result["config"] = request_json("/config", api_prefix=V7_PREFIX)
    except RuntimeError as error:
        result["errors"].append({"operation": "config", "message": str(error)})

    if result["subjects"]:
        subject = result["subjects"][0]
        encoded_subject = quote(subject, safe="")
        try:
            versions = request_json(f"/subjects/{encoded_subject}/versions", api_prefix=V7_PREFIX) or []
            result["sampleSubject"] = subject
            result["versions"] = versions
            if versions:
                latest = request_json(
                    f"/subjects/{encoded_subject}/versions/latest",
                    api_prefix=V7_PREFIX,
                )
                result["latestSchema"] = latest
        except RuntimeError as error:
            result["errors"].append({"operation": "subject_schema", "message": str(error)})

    return result


v7_probe = probe_confluent_v7()
show_result("Confluent-compatible v7 probe:", v7_probe)


WARNING Attempt 1 failed for http://localhost:8081/apis/ccompat/v7/subjects: HTTP 404 for http://localhost:8081/apis/ccompat/v7/subjects: {"error_code":404,"message":"HTTP 404 Not Found"}
WARNING Attempt 2 failed for http://localhost:8081/apis/ccompat/v7/subjects: HTTP 404 for http://localhost:8081/apis/ccompat/v7/subjects: {"error_code":404,"message":"HTTP 404 Not Found"}
WARNING Attempt 3 failed for http://localhost:8081/apis/ccompat/v7/subjects: HTTP 404 for http://localhost:8081/apis/ccompat/v7/subjects: {"error_code":404,"message":"HTTP 404 Not Found"}


Confluent-compatible v7 probe:
{
  "apiVersion": "v7",
  "apiPrefix": "/apis/ccompat/v7",
  "compatible": false,
  "subjects": [],
  "config": null,
  "errors": [
    {
      "operation": "subjects",
      "message": "Registry request failed: http://localhost:8081/apis/ccompat/v7/subjects"
    }
  ]
}


## 4. List Core v3 Groups

The UI hierarchy starts with groups. This uses only `GET /apis/registry/v3/groups`.

In [14]:
try:
    groups_response = request_json("/groups", api_prefix=API_VARIANTS["core_v3"])
    groups = groups_response if isinstance(groups_response, list) else groups_response.get("groups", [])
    show_result("Core v3 groups:", groups)
except RuntimeError as error:
    groups = []
    print(error)


WARNING Attempt 1 failed for http://localhost:8081/apis/registry/v3/groups: HTTP 404 for http://localhost:8081/apis/registry/v3/groups: {"error_code":404,"message":"HTTP 404 Not Found"}
WARNING Attempt 2 failed for http://localhost:8081/apis/registry/v3/groups: HTTP 404 for http://localhost:8081/apis/registry/v3/groups: {"error_code":404,"message":"HTTP 404 Not Found"}
WARNING Attempt 3 failed for http://localhost:8081/apis/registry/v3/groups: HTTP 404 for http://localhost:8081/apis/registry/v3/groups: {"error_code":404,"message":"HTTP 404 Not Found"}


Registry request failed: http://localhost:8081/apis/registry/v3/groups


## 5. List Artifacts for Each Group

For every group, call `GET /apis/registry/v3/groups/{groupId}/artifacts`. This produces the second level: `groups -> artifacts`.

In [6]:
def first_value(record: dict[str, Any], *keys: str, default: Any = None) -> Any:
    for key in keys:
        if record.get(key) is not None:
            return record[key]
    return default


def group_id_from_item(item: Any) -> str:
    if isinstance(item, str):
        return item
    return str(first_value(item, "groupId", "id", "name", default=""))


def normalize_artifact(item: dict[str, Any], group_id: str) -> dict[str, Any]:
    return {
        "groupId": group_id,
        "artifactId": first_value(item, "artifactId", "id"),
        "name": first_value(item, "name", "artifactId", "id"),
        "description": first_value(item, "description", default=""),
        "createdAt": first_value(item, "createdAt", "createdOn"),
        "type": first_value(item, "type", "artifactType"),
        "labels": first_value(item, "labels", default={}),
    }


def list_group_artifacts(group_id: str) -> list[dict[str, Any]]:
    encoded_group = quote(group_id, safe="")
    result = request_json(
        f"/groups/{encoded_group}/artifacts",
        api_prefix=API_VARIANTS["core_v3"],
    )
    items = result if isinstance(result, list) else (result or {}).get("artifacts", [])
    return [normalize_artifact(item, group_id) for item in items if isinstance(item, dict)]


artifacts_by_group: dict[str, list[dict[str, Any]]] = {}
for group_item in groups:
    group_id = group_id_from_item(group_item)
    if not group_id:
        continue
    try:
        artifacts_by_group[group_id] = list_group_artifacts(group_id)
    except RuntimeError as error:
        LOGGER.warning("Could not list artifacts for group %s: %s", group_id, error)
        artifacts_by_group[group_id] = []

normalized_artifacts = [
    artifact
    for group_artifacts in artifacts_by_group.values()
    for artifact in group_artifacts
]
show_result("Artifacts by group:", artifacts_by_group)


Artifacts by group:
{}


## 6. List Versions and Read Schema Content

For each artifact, call `GET /groups/{groupId}/artifacts/{artifactId}/versions`, then fetch each version's content with `GET /groups/{groupId}/artifacts/{artifactId}/versions/{version}/content`.

In [7]:
def list_artifact_versions(group_id: str, artifact_id: str) -> list[Any]:
    encoded_group = quote(group_id, safe="")
    encoded_artifact = quote(artifact_id, safe="")
    result = request_json(
        f"/groups/{encoded_group}/artifacts/{encoded_artifact}/versions",
        api_prefix=API_VARIANTS["core_v3"],
    )
    return result if isinstance(result, list) else (result or {}).get("versions", [])


def version_value(version_item: Any) -> str:
    if isinstance(version_item, str):
        return version_item
    return str(first_value(version_item, "version", "versionId", default=""))


def fetch_version_content(group_id: str, artifact_id: str, version: str) -> Any:
    encoded_group = quote(group_id, safe="")
    encoded_artifact = quote(artifact_id, safe="")
    encoded_version = quote(version, safe="")
    return request_json(
        f"/groups/{encoded_group}/artifacts/{encoded_artifact}/versions/{encoded_version}/content",
        api_prefix=API_VARIANTS["core_v3"],
    )


def build_registry_tree() -> list[dict[str, Any]]:
    tree = []
    for group_item in groups:
        group_id = group_id_from_item(group_item)
        if not group_id:
            continue
        group_node = {"groupId": group_id, "artifacts": []}
        for artifact in artifacts_by_group.get(group_id, []):
            artifact_id = artifact.get("artifactId")
            if not artifact_id:
                continue
            artifact_node = {**artifact, "versions": []}
            try:
                versions = list_artifact_versions(group_id, artifact_id)
            except RuntimeError as error:
                LOGGER.warning("Could not list versions for %s/%s: %s", group_id, artifact_id, error)
                versions = []
            for version_item in versions:
                version = version_value(version_item)
                if not version:
                    continue
                try:
                    content = fetch_version_content(group_id, artifact_id, version)
                except RuntimeError as error:
                    LOGGER.warning("Could not fetch content for %s/%s/%s: %s", group_id, artifact_id, version, error)
                    content = None
                artifact_node["versions"].append({
                    "version": version,
                    "content": content,
                })
            group_node["artifacts"].append(artifact_node)
        tree.append(group_node)
    return tree


registry_tree = build_registry_tree()
show_result("Registry hierarchy:", registry_tree)


Registry hierarchy:
[]


## 7. Format Data for UI Display

The UI-ready structure is:

`groups -> artifacts -> versions -> content`

The v7 result remains available for common Confluent-compatible operations.

In [8]:
ui_payload = {
    "provider": "apicurio",
    "apiVersion": "v3",
    "registryUrl": REGISTRY_URL,
    "reachable": bool(groups is not None),
    "hierarchy": "groups > artifacts > versions > content",
    "groups": registry_tree,
    "compatibility": {
        "confluentV7": v7_probe,
    },
}

show_result("UI-ready registry payload:", ui_payload)


NameError: name 'v7_probe' is not defined

## 8. Handle Errors and Retry Logic

The request helper retries transient failures and includes the exact API prefix, URL, status code, and response excerpt in the error. A UI integration can map these failures to friendly connection or permission messages.

In [9]:
def run_probe() -> dict[str, Any]:
    """Run the GET-only hierarchy probe and include v7 compatibility status."""
    payload = {
        "provider": "apicurio",
        "registryUrl": REGISTRY_URL,
        "apiVersion": "v3",
        "hierarchy": "groups > artifacts > versions > content",
        "compatibility": {"confluentV7": probe_confluent_v7()},
        "reachable": False,
        "groups": [],
    }
    try:
        root_groups = request_json("/groups", api_prefix=API_VARIANTS["core_v3"])
        payload["reachable"] = True
        payload["groups"] = registry_tree if root_groups is not None else []
    except RuntimeError as error:
        payload["error"] = str(error)
    return payload


probe_payload = run_probe()
show_result("Final GET-only probe payload:", probe_payload)


NameError: name 'probe_confluent_v7' is not defined